In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import pandas as pd
import muon as mu
import scanpy as sc
try:
    import scirpy as ir
except ImportError:
    pass  # scirpy is optional
np.random.seed(42)
import random
random.seed(42)

ModuleNotFoundError: No module named 'scirpy'

In [ ]:
filename = ".\data\EAE\CCA\merged_EAE_CD4"
mdata_ori = mu.read(filename + ".h5mu")
mdata = mdata_ori.copy()

# Keep a copy of original data for comparison
mdata_original = mdata_ori.copy()
mdata

In [ ]:
# Check the structure of mdata and available batches
print("Modalities:", list(mdata.mod.keys()))
print("\nBatch information (GSE):")
print(mdata.obs['GSE'].value_counts())
print(f"\nTotal cells: {mdata.n_obs}")
print(f"Total features: {mdata.n_vars}")


In [ ]:
# Apply Domain-Specific Batch Normalization (DSBN)
# Normalize each batch (GSE) separately to reduce batch effects

from sklearn.preprocessing import StandardScaler
import scipy.sparse as sp

def apply_dsbn(mdata, batch_key='GSE', modality='gex', layer=None):
    """
    Apply Domain-Specific Batch Normalization to mdata.
    
    Parameters:
    -----------
    mdata : MuData
        MuData object containing the data
    batch_key : str
        Key in mdata.obs that identifies batches
    modality : str
        Modality to normalize (default: 'gex')
    layer : str or None
        Layer to normalize (None means use .X)
    """
    adata = mdata[modality]
    
    # Get batch labels
    batches = adata.obs[batch_key].values
    unique_batches = np.unique(batches)
    
    print(f"Applying DSBN to {modality} modality")
    print(f"Number of batches: {len(unique_batches)}")
    print(f"Batches: {unique_batches}")
    
    # Get the data matrix
    if layer is not None:
        X = adata.layers[layer].copy()
    else:
        X = adata.X.copy()
    
    # Check if sparse
    is_sparse = sp.issparse(X)
    if is_sparse:
        X = X.toarray()
    
    # Normalize each batch separately
    X_normalized = np.zeros_like(X)
    
    for batch in unique_batches:
        batch_mask = batches == batch
        batch_indices = np.where(batch_mask)[0]
        
        # Extract batch data
        X_batch = X[batch_indices, :]
        
        # Standardize this batch
        scaler = StandardScaler()
        X_batch_normalized = scaler.fit_transform(X_batch)
        
        # Store normalized batch
        X_normalized[batch_indices, :] = X_batch_normalized
        
        print(f"  Batch {batch}: {len(batch_indices)} cells normalized")
    
    # Convert back to sparse if original was sparse
    if is_sparse:
        X_normalized = sp.csr_matrix(X_normalized)
    
    # Store normalized data
    if layer is not None:
        adata.layers[layer] = X_normalized
    else:
        adata.X = X_normalized
    
    print(f"\nDSBN completed. Normalized data stored in {modality}.X")
    return mdata

# Apply DSBN
mdata = apply_dsbn(mdata, batch_key='GSE', modality='gex')


In [ ]:
# Verify the normalization
import scipy.sparse as sp

print("Verification of DSBN:")
print(f"Data shape: {mdata['gex'].X.shape}")
print(f"Data type: {type(mdata['gex'].X)}")

# Check statistics per batch
gex_adata = mdata['gex']
for batch in gex_adata.obs['GSE'].unique():
    batch_mask = gex_adata.obs['GSE'] == batch
    X_batch = gex_adata[batch_mask].X
    
    if sp.issparse(X_batch):
        X_batch = X_batch.toarray()
    
    print(f"\nBatch {batch}:")
    print(f"  Mean: {X_batch.mean():.6f} (should be ~0)")
    print(f"  Std: {X_batch.std():.6f} (should be ~1)")
    print(f"  Min: {X_batch.min():.6f}, Max: {X_batch.max():.6f}")


In [ ]:
# UMAP comparison: Original vs DSBN-normalized data

# Set scanpy settings
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=80, facecolor='white')

# Prepare original data for UMAP (if not already processed)
print("Computing UMAP for original data (without DSBN)...")
mdata_orig_gex = mdata_original['gex'].copy()

# Compute PCA and UMAP for original data
sc.pp.pca(mdata_orig_gex, svd_solver='arpack', n_comps=50)
sc.pp.neighbors(mdata_orig_gex, n_neighbors=50, n_pcs=50)
sc.tl.umap(mdata_orig_gex, min_dist=0.5, spread=1.0)

print("Computing UMAP for DSBN-normalized data...")
mdata_dsbn_gex = mdata['gex'].copy()

# Compute PCA and UMAP for DSBN-normalized data
sc.pp.pca(mdata_dsbn_gex, svd_solver='arpack', n_comps=50)
sc.pp.neighbors(mdata_dsbn_gex, n_neighbors=50, n_pcs=50)
sc.tl.umap(mdata_dsbn_gex, min_dist=0.5, spread=1.0)

print("UMAP computation completed!")


In [ ]:
# Visualize UMAP comparison: Original vs DSBN-normalized, colored by GSE batch

# Extract UMAP coordinates
umap_orig = mdata_orig_gex.obsm['X_umap']
umap_dsbn = mdata_dsbn_gex.obsm['X_umap']

# Get batch labels
batch_labels_orig = mdata_orig_gex.obs['GSE'].values
batch_labels_dsbn = mdata_dsbn_gex.obs['GSE'].values

# Get unique batches for consistent coloring
unique_batches = np.unique(batch_labels_orig)
colors = plt.cm.tab10(np.linspace(0, 1, len(unique_batches)))
batch_color_map = {batch: colors[i] for i, batch in enumerate(unique_batches)}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Original data (without DSBN)
ax1 = axes[0]
for batch in unique_batches:
    mask = batch_labels_orig == batch
    ax1.scatter(umap_orig[mask, 0], umap_orig[mask, 1], 
                c=batch_color_map[batch], label=batch, s=1, alpha=0.6)
ax1.set_xlabel('UMAP 1', fontsize=12)
ax1.set_ylabel('UMAP 2', fontsize=12)
ax1.set_title('Original Data (No DSBN)', fontsize=14, fontweight='bold')
ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax1.grid(True, alpha=0.3)

# DSBN-normalized data
ax2 = axes[1]
for batch in unique_batches:
    mask = batch_labels_dsbn == batch
    ax2.scatter(umap_dsbn[mask, 0], umap_dsbn[mask, 1], 
                c=batch_color_map[batch], label=batch, s=1, alpha=0.6)
ax2.set_xlabel('UMAP 1', fontsize=12)
ax2.set_ylabel('UMAP 2', fontsize=12)
ax2.set_title('DSBN-Normalized Data', fontsize=14, fontweight='bold')
ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nComparison complete!")
print("If DSBN is effective, batches should be better mixed in the right panel.")


In [ ]:
# Additional comparison: Check if other annotations are preserved
# This helps verify that DSBN reduces batch effects without losing biological signal

# Check available annotations
available_annotations = [col for col in mdata_orig_gex.obs.columns 
                        if col not in ['GSE'] and mdata_orig_gex.obs[col].dtype in ['object', 'category']]

if len(available_annotations) > 0:
    # Use the first available annotation (or 'cell_type' if available)
    annotation_key = 'cell_type' if 'cell_type' in available_annotations else available_annotations[0]
    
    print(f"Comparing with annotation: {annotation_key}")
    
    # Extract annotation labels
    annot_orig = mdata_orig_gex.obs[annotation_key].values
    annot_dsbn = mdata_dsbn_gex.obs[annotation_key].values
    
    # Get unique annotations for coloring
    unique_annot = np.unique(annot_orig)
    annot_colors = plt.cm.Set3(np.linspace(0, 1, len(unique_annot)))
    annot_color_map = {annot: annot_colors[i] for i, annot in enumerate(unique_annot)}
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    # Original: GSE (batch)
    ax1 = axes[0, 0]
    for batch in unique_batches:
        mask = batch_labels_orig == batch
        ax1.scatter(umap_orig[mask, 0], umap_orig[mask, 1], 
                    c=batch_color_map[batch], label=batch, s=1, alpha=0.6)
    ax1.set_title('Original: GSE (batch)', fontsize=12, fontweight='bold')
    ax1.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7)
    ax1.grid(True, alpha=0.3)
    
    # Original: Annotation
    ax2 = axes[0, 1]
    for annot in unique_annot:
        mask = annot_orig == annot
        ax2.scatter(umap_orig[mask, 0], umap_orig[mask, 1], 
                    c=annot_color_map[annot], label=annot, s=1, alpha=0.6)
    ax2.set_title(f'Original: {annotation_key}', fontsize=12, fontweight='bold')
    ax2.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7)
    ax2.grid(True, alpha=0.3)
    
    # DSBN: GSE (batch)
    ax3 = axes[1, 0]
    for batch in unique_batches:
        mask = batch_labels_dsbn == batch
        ax3.scatter(umap_dsbn[mask, 0], umap_dsbn[mask, 1], 
                    c=batch_color_map[batch], label=batch, s=1, alpha=0.6)
    ax3.set_title('DSBN: GSE (batch)', fontsize=12, fontweight='bold')
    ax3.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7)
    ax3.grid(True, alpha=0.3)
    
    # DSBN: Annotation
    ax4 = axes[1, 1]
    for annot in unique_annot:
        mask = annot_dsbn == annot
        ax4.scatter(umap_dsbn[mask, 0], umap_dsbn[mask, 1], 
                    c=annot_color_map[annot], label=annot, s=1, alpha=0.6)
    ax4.set_title(f'DSBN: {annotation_key}', fontsize=12, fontweight='bold')
    ax4.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=7)
    ax4.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    print(f"\n{annotation_key} structure should be preserved while batch effects are reduced.")
else:
    print("No additional categorical annotations found. Skipping annotation comparison.")
